In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Load your data
# Make sure your CSV files are in a folder named 'data'
teams = pd.read_csv('data/MTeams.csv')
season_stats = pd.read_csv('data/MRegularSeasonDetailedResults.csv')
tourney_results = pd.read_csv('data/MNCAATourneyCompactResults.csv')

print("Step 1 Complete: Data loaded successfully.")

Step 1 Complete: Data loaded successfully.


In [33]:
print("--- 2026 First Round Predictions ---")

# Ensure bracket_matchups exists before running
if 'bracket_matchups' in locals():
    for team1, team2 in bracket_matchups:
        try:
            prob = predict_game(team1, team2)
            if prob is not None:
                print(f"{team1} vs {team2}: {prob:.2%} win chance for {team1}")
            else:
                print(f"Skipping {team1} vs {team2}: Stats not found.")
        except Exception as e:
            print(f"Error predicting {team1} vs {team2}: {e}")
else:
    print("Error: 'bracket_matchups' is not defined. Please run the cell that creates your matchup list.")

--- 2026 First Round Predictions ---
Duke vs Siena: 77.42% win chance for Duke
Ohio State vs TCU: 60.86% win chance for Ohio State
St. John's vs Northern Iowa: 71.54% win chance for St. John's
Kansas vs Cal Baptist: 81.91% win chance for Kansas
Louisville vs South Florida: 75.30% win chance for Louisville
Michigan State vs North Dakota State: 76.24% win chance for Michigan State
UCLA vs UCF: 71.76% win chance for UCLA
UConn vs Furman: 67.36% win chance for UConn
Arizona vs LIU: 74.56% win chance for Arizona
Villanova vs Utah State: 40.21% win chance for Villanova
Wisconsin vs High Point: 44.67% win chance for Wisconsin
Arkansas vs Hawaii: 64.71% win chance for Arkansas
BYU vs Texas: 68.37% win chance for BYU
Gonzaga vs Kennesaw State: 90.10% win chance for Gonzaga
Miami (Florida) vs Missouri: 45.55% win chance for Miami (Florida)
Michigan vs UMBC: 62.62% win chance for Michigan
Texas Tech vs Akron: 50.30% win chance for Texas Tech
Alabama vs Hofstra: 55.08% win chance for Alabama
Tenne

In [17]:
def predict_game(team1_name, team2_name):
    # 1. Map names using your master dictionary, or keep the original if not in map
    t1_lookup_name = name_map.get(team1_name, team1_name)
    t2_lookup_name = name_map.get(team2_name, team2_name)

    # 2. Get the IDs from the teams dataframe
    # We use .get() to avoid errors if the name isn't found
    t1_data = teams[teams['TeamName'] == t1_lookup_name]
    t2_data = teams[teams['TeamName'] == t2_lookup_name]

    # 3. If either team is empty, return None to trigger the "STILL MISSING" message
    if t1_data.empty or t2_data.empty:
        return None

    t1_id = t1_data['TeamID'].values[0]
    t2_id = t2_data['TeamID'].values[0]

    # 4. Get the stats using the ID as an index
    s1 = team_stats.loc[t1_id]
    s2 = team_stats.loc[t2_id]

    # 5. Calculate differences and predict
    diff = pd.DataFrame([[
        s1['Score'] - s2['Score'],
        s1['FGM'] - s2['FGM'],
        s1['Ast'] - s2['Ast']
    ]], columns=['ScoreDiff', 'FGM_Diff', 'Ast_Diff'])

    return model.predict_proba(diff)[0][1]

In [18]:
print(season_stats.columns)

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='str')


In [19]:
# 1. Isolate and rename winning team stats
winners = season_stats[['WTeamID', 'WScore', 'WFGM', 'WAst']].copy()
winners.columns = ['TeamID', 'Score', 'FGM', 'Ast']

# 2. Isolate and rename losing team stats
losers = season_stats[['LTeamID', 'LScore', 'LFGM', 'LAst']].copy()
losers.columns = ['TeamID', 'Score', 'FGM', 'Ast']

# 3. Combine both into one master list
# This creates a dataset where every row is a performance by a single team
all_perf = pd.concat([winners, losers])

# 4. Group by TeamID to get the average stats for every team
team_stats = all_perf.groupby('TeamID').mean()

print("Stats summarized! You can now look up any team using its TeamID.")

Stats summarized! You can now look up any team using its TeamID.


In [20]:
def predict_game(team1_name, team2_name):
    t1_lookup = teams[teams['TeamName'] == team1_name]
    t2_lookup = teams[teams['TeamName'] == team2_name]

    if t1_lookup.empty or t2_lookup.empty:
        return None

    t1_id = t1_lookup['TeamID'].values[0]
    t2_id = t2_lookup['TeamID'].values[0]

    # Use .loc to get the stats from your new table
    s1 = team_stats.loc[t1_id]
    s2 = team_stats.loc[t2_id]

    diff = pd.DataFrame([[
        s1['Score'] - s2['Score'],
        s1['FGM'] - s2['FGM'],
        s1['Ast'] - s2['Ast']
    ]], columns=['ScoreDiff', 'FGM_Diff', 'Ast_Diff'])

    return model.predict_proba(diff)[0][1]

In [21]:
# 1. Create a list to store training rows
training_data = []
results = []

for _, row in tourney_results.iterrows():
    w_id = row['WTeamID']
    l_id = row['LTeamID']

    # Get stats
    w_stats = team_stats.loc[w_id]
    l_stats = team_stats.loc[l_id]

    # CASE A: Winner is Team 1, Loser is Team 2 (This is a Win: 1)
    training_data.append([
        w_stats['Score'] - l_stats['Score'],
        w_stats['FGM'] - l_stats['FGM'],
        w_stats['Ast'] - l_stats['Ast']
    ])
    results.append(1)

    # CASE B: Loser is Team 1, Winner is Team 2 (This is a Loss: 0)
    # We swap them to show the model what a loss looks like
    training_data.append([
        l_stats['Score'] - w_stats['Score'],
        l_stats['FGM'] - w_stats['FGM'],
        l_stats['Ast'] - w_stats['Ast']
    ])
    results.append(0)

# Now X and y have both 1s and 0s!
X = pd.DataFrame(training_data, columns=['ScoreDiff', 'FGM_Diff', 'Ast_Diff'])
y = pd.Series(results)

In [22]:
# Train the model
model = LogisticRegression()
model.fit(X, y)

print("Model successfully trained on balanced data!")

Model successfully trained on balanced data!


In [24]:
print(sorted(teams['TeamName'].unique()))

['Abilene Chr', 'Air Force', 'Akron', 'Alabama', 'Alabama A&M', 'Alabama St', 'Alcorn St', 'Alliant Intl', 'American Univ', 'Appalachian St', 'Arizona', 'Arizona St', 'Ark Little Rock', 'Ark Pine Bluff', 'Arkansas', 'Arkansas St', 'Armstrong St', 'Army', 'Auburn', 'Augusta', 'Austin Peay', 'BYU', 'Ball St', 'Baylor', 'Bellarmine', 'Belmont', 'Bethune-Cookman', 'Binghamton', 'Birmingham So', 'Boise St', 'Boston College', 'Boston Univ', 'Bowling Green', 'Bradley', 'Brooklyn', 'Brown', 'Bryant', 'Bucknell', 'Buffalo', 'Butler', 'C Michigan', 'CS Bakersfield', 'CS Fullerton', 'CS Northridge', 'CS Sacramento', 'Cal Baptist', 'Cal Poly', 'California', 'Campbell', 'Canisius', 'Cent Arkansas', 'Centenary', 'Central Conn', 'Charleston So', 'Charlotte', 'Chattanooga', 'Chicago St', 'Cincinnati', 'Citadel', 'Clemson', 'Cleveland St', 'Coastal Car', 'Col Charleston', 'Colgate', 'Colorado', 'Colorado St', 'Columbia', 'Connecticut', 'Coppin St', 'Cornell', 'Creighton', 'Dartmouth', 'Davidson', 'Dayt

In [ ]:
def predict_game(team1_name, team2_name):
    # Map the names using the dictionary, or keep the original if not found
    t1_lookup_name = name_map.get(team1_name, team1_name)
    t2_lookup_name = name_map.get(team2_name, team2_name)

    # Now use the mapped names to look up in your 'teams' table
    t1_lookup = teams[teams['TeamName'] == t1_lookup_name]
    t2_lookup = teams[teams['TeamName'] == team2_name] # You should map t2_lookup_name too!

    # ... rest of your code ...

In [ ]:
def predict_game(team1_name, team2_name):
    # Use .get() to look up the official name, or default to the original if not in the map
    t1_lookup_name = name_map.get(team1_name, team1_name)
    t2_lookup_name = name_map.get(team2_name, team2_name)

    t1_lookup = teams[teams['TeamName'] == t1_lookup_name]
    t2_lookup = teams[teams['TeamName'] == t2_lookup_name] # Fixed: now mapping both!

    if t1_lookup.empty or t2_lookup.empty:
        return None

    t1_id = t1_lookup['TeamID'].values[0]
    t2_id = t2_lookup['TeamID'].values[0]

    s1 = team_stats.loc[t1_id]
    s2 = team_stats.loc[t2_id]

    diff = pd.DataFrame([[
        s1['Score'] - s2['Score'],
        s1['FGM'] - s2['FGM'],
        s1['Ast'] - s2['Ast']
    ]], columns=['ScoreDiff', 'FGM_Diff', 'Ast_Diff'])

    return model.predict_proba(diff)[0][1]

In [30]:
name_map = {
    'LIU': 'LIU Brooklyn',
    'Kennesaw State': 'Kennesaw',
    'Texas A&M': 'Texas A&M',
    'Ohio State': 'Ohio St',
    'Michigan State': 'Michigan St',
    'North Dakota State': 'North Dakota',
    "St. John's": "St John's",
    'UConn': 'Connecticut',
    'Utah State': 'Utah St',
    'Miami (Florida)': 'Miami FL',
    'Wright State': 'Wright St',
    'Iowa State': 'Iowa St',
    'Tennessee State': 'Tennessee St',
    'Prairie View A&M': 'Prairie View',
    'McNeese': 'McNeese St',
    'Arizona': 'Arizona',
    'Purdue': 'Purdue',
    'Georgia': 'Georgia',
    'Gonzaga': 'Gonzaga',
    'Northern Iowa': 'Northern Iowa'
}

In [ ]:
print("--- 2026 First Round Predictions (Final) ---")

for team1, team2 in bracket_matchups:
    prob = predict_game(team1, team2)
    if prob is not None:
        print(f"{team1} vs {team2}: {prob:.2%} win chance for {team1}")
    else:
        # If a team is still missing, this will tell you exactly which one it is
        print(f"STILL MISSING: '{team1}' or '{team2}'")

In [32]:
def predict_game(team1_name, team2_name):
    # Normalize by stripping whitespace
    t1_lookup = name_map.get(team1_name.strip(), team1_name.strip())
    t2_lookup = name_map.get(team2_name.strip(), team2_name.strip())

    # Check existence
    if t1_lookup not in teams['TeamName'].values:
        print(f"DEBUG: '{t1_lookup}' (from '{team1_name}') NOT FOUND in database!")
        return None
    if t2_lookup not in teams['TeamName'].values:
        print(f"DEBUG: '{t2_lookup}' (from '{team2_name}') NOT FOUND in database!")
        return None

    # Proceed with prediction
    t1_id = teams[teams['TeamName'] == t1_lookup]['TeamID'].values[0]
    t2_id = teams[teams['TeamName'] == t2_lookup]['TeamID'].values[0]

    s1 = team_stats.loc[t1_id]
    s2 = team_stats.loc[t2_id]

    diff = pd.DataFrame([[s1['Score'] - s2['Score'], s1['FGM'] - s2['FGM'], s1['Ast'] - s2['Ast']]],
                        columns=['ScoreDiff', 'FGM_Diff', 'Ast_Diff'])
    return model.predict_proba(diff)[0][1]

In [29]:
bracket_matchups = [
    ("Duke", "Siena"),
    ("Ohio State", "TCU"),
    ("St. John's", "Northern Iowa"),
    ("Kansas", "Cal Baptist"),
    ("Louisville", "South Florida"),
    ("Michigan State", "North Dakota State"),
    ("UCLA", "UCF"),
    ("UConn", "Furman"),
    ("Arizona", "LIU"),
    ("Villanova", "Utah State"),
    ("Wisconsin", "High Point"),
    ("Arkansas", "Hawaii"),
    ("BYU", "Texas"),
    ("Gonzaga", "Kennesaw State"),
    ("Miami (Florida)", "Missouri"),
    ("Michigan", "UMBC"),
    ("Texas Tech", "Akron"),
    ("Alabama", "Hofstra"),
    ("Tennessee", "SMU"),
    ("Virginia", "Wright State"),
    ("Kentucky", "Santa Clara"),
    ("Iowa State", "Tennessee State"),
    ("Florida", "Prairie View A&M"),
    ("Clemson", "Iowa"),
    ("Vanderbilt", "McNeese"),
    ("Nebraska", "Troy"),
    ("North Carolina", "VCU"),
    ("Illinois", "Penn"),
    ("Houston", "Idaho")
]